In [ ]:
import sys
import os
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))
import random
import pandas as pd
from datetime import datetime, timedelta
from data_simulation.gas_turbine import GasTurbine
from data_simulation.compressor import Compressor
from data_simulation.pump import Pump
from data_simulation.physics.environmental_conditions import EnvironmentalConditions, LocationType
from data_simulation.physics.weather_api_client import create_hybrid_environment
from data_simulation.ml_utils.ml_output_modes import OutputMode
from ingestion.equipment_sim import simulate_equipment
from ingestion.data_pipeline import DataPipeline
from ingestion.db_setup import Database
from ingestion.bulk_insert import bulk_insert_telemetry, insert_failures, insert_maintenance
import sqlite3
from sqlalchemy import create_engine, text

In [ ]:
# Simulation Configuration
CONFIG = {
    # Simulation time
    'start_date': datetime(2025, 7, 1),
    'duration_days': 180,
    'sample_interval_min': 5,
    
    # Equipment counts
    'n_turbines': 15,
    'n_compressors': 15,
    'n_pumps': 30,
    
    # Degradation
    'degradation_multiplier': 1.7,
    
    # Initial health ranges per equipment type
    'health_ranges': {
        'turbine': {
            'hgp': (0.70, 0.98),
            'blade': (0.70, 0.98),
            'bearing': (0.70, 0.98),
            'fuel': (0.70, 0.98),
        },
        'compressor': {
            'impeller': (0.88, 0.98),
            'bearing': (0.85, 0.98),
            'seal_primary': (0.85, 0.98),
            'seal_secondary': (0.85, 0.98),
        },
        'pump': {
            'impeller': (0.70, 0.98),
            'seal': (0.70, 0.98),
            'bearing_de': (0.70, 0.98),
            'bearing_nde': (0.70, 0.98),
        }
    }
}

print("Simulation Configuration:")
for key, value in CONFIG.items():
    if key != 'health_ranges':
        print(f"  {key}: {value}")
print(f"  Health ranges: {len(CONFIG['health_ranges'])} equipment types configured")

In [ ]:
def generate_initial_health(config: dict, equipment_type: str) -> dict:
    """Generate random initial health values for equipment based on configured ranges."""
    ranges = config['health_ranges'][equipment_type]
    return {
        component: round(random.uniform(lo, hi), 3)
        for component, (lo, hi) in ranges.items()
    }

In [ ]:
# Create environmental model for Bonny Island using REAL weather data
print("\nSetting up Weather API for Bonny Island...")
print("Note: This uses real weather data. Set use_real_weather=False for synthetic data.")

# Create hybrid environment with real weather and synthetic fallback
fallback = EnvironmentalConditions(LocationType.TROPICAL)

bonny_island_env = create_hybrid_environment(
use_real_weather=True,
    api_provider="weatherapi",
    api_key = os.getenv('WEATHER_API_KEY'),
    location_name="Port Harcourt",  
    country="Nigeria",
    fallback_source=fallback,  
    cache_enabled=True
)
# This fetches hourly weather once, then uses cached data for 5-min interpolation
print(f"Pre-caching weather data for {CONFIG['duration_days']} days...")
bonny_island_env.preload_cache(
    start_date=CONFIG['start_date'],
    end_date=CONFIG['start_date'] + timedelta(days=CONFIG['duration_days']),
    interval_hours=1  
)

# Test environmental conditions
test_conditions = bonny_island_env.get_conditions(
    elapsed_hours=0, 
    timestamp=CONFIG['start_date']
)
print("\nBonny Island Environmental Conditions (July 1, 2025):")
print(f"  Temperature: {test_conditions['ambient_temp_C']:.1f}°C (REAL DATA)")
print(f"  Humidity: {test_conditions['humidity_percent']:.1f}%")
print(f"  Pressure: {test_conditions['pressure_kPa']:.1f} kPa")
# print(f"  Wind Speed: {test_conditions['wind_speed_m_s']:.1f} m/s")
print(f"  Corrosion factor: {test_conditions['corrosion_factor']:.2f}x (coastal)")
print(f"  Fouling factor: {test_conditions['fouling_factor']:.2f}x")

In [ ]:
conn = sqlite3.connect("weather_cache_Port_Harcourt.db")

env_model = pd.read_sql_query(
    "SELECT * FROM weather_cache",
    conn
)
conn.close()
# Convert timestamps
env_model["timestamp"] = pd.to_datetime(env_model["timestamp"], utc=True)

# Select July 1, 2025 at 00:00
row = env_model.loc[env_model["timestamp"] == "2025-12-25T14:00:00Z"].iloc[0]

# print("\nBonny Island Environmental Conditions (July 1, 2025):")
print(f"  Temperature: {row['ambient_temp_C']:.1f}°C (REAL DATA)")
print(f"  Humidity: {row['humidity_percent']:.1f}%")
print(f"  Pressure: {row['pressure_kPa']:.1f} kPa")

In [ ]:
# Seed master data for all equipment using DataPipeline
print("SEEDING MASTER DATA FOR ALL EQUIPMENT")

try:
    # Initialize data pipeline
    db_url = os.getenv('POSTGRES_URL')
    pipeline = DataPipeline(db_url)
    pipeline.connect()
    
    print("\nDatabase connection successful\n")
    
    # Seed master data
    turbine_ids, compressor_ids, pump_ids = pipeline.seed_master_data(
        turbine_count=CONFIG['n_turbines'],
        compressor_count=CONFIG['n_compressors'],
        pump_count=CONFIG['n_pumps']
    )
    
    print("MASTER DATA SEEDING COMPLETE")
    print(f"\nTotal equipment created: {len(turbine_ids) + len(compressor_ids) + len(pump_ids)}")
    print(f"  - Gas Turbines: {len(turbine_ids)} (IDs: {turbine_ids[:5]}{'...' if len(turbine_ids) > 5 else ''})")
    print(f"  - Compressors: {len(compressor_ids)} (IDs: {compressor_ids[:5]}{'...' if len(compressor_ids) > 5 else ''})")
    print(f"  - Pumps: {len(pump_ids)} (IDs: {pump_ids[:6]}{'...' if len(pump_ids) > 6 else ''})")
    
    # Retrieve and display sample master data
    print("SAMPLE MASTER DATA FROM DATABASE")
    
    # Get turbine configs
    turbine_configs = pipeline.master_data.get_configs(turbine_ids[:3], 'turbine')
    print("\nGas Turbines (first 3):")
    df_turbines = pd.DataFrame(turbine_configs)
    display(df_turbines)
    
    # Get compressor configs
    compressor_configs = pipeline.master_data.get_configs(compressor_ids[:3], 'compressor')
    print("\nCompressors (first 3):")
    df_compressors = pd.DataFrame(compressor_configs)
    display(df_compressors)
    
    # Get pump configs
    pump_configs = pipeline.master_data.get_configs(pump_ids[:3], 'pump')
    print("\nPumps (first 3):")
    df_pumps = pd.DataFrame(pump_configs)
    display(df_pumps)
    
    # Close database connection
    pipeline.close()
    print("\nDatabase connection closed")
    
except Exception as e:
    print(f"\nError: {e}")
    print("Please ensure PostgreSQL is running and credentials are correct")
    raise

In [ ]:
print(f"\nSimulating {CONFIG['n_turbines']} Gas Turbines...")
turbine_telemetry = []
turbine_telemetry_raw = []  # Raw records with nested state for bulk_insert
turbine_failures = []
turbine_maintenance = []

for i in range(CONFIG['n_turbines']):
    turbine_id = i + 1
    
    # Create turbine with initial health
    initial_health = generate_initial_health(CONFIG, 'turbine')
    print(f"GT-{turbine_id:03d} initial health: {initial_health}")
    
    turbine = GasTurbine(
        name=f"GT-{turbine_id:03d}",
        initial_health=initial_health,
        env_model=env_model,
        enable_environmental=True,
        enable_maintenance=True,  
        enable_thermal_transients=True,
        enable_enhanced_vibration=True,
        output_mode = OutputMode.DERIVED_FEATURES
    )
    
    # Simulate with maintenance/repair after failures
    for record in simulate_equipment(
        turbine, 
        turbine_id, 
        'turbine',
        CONFIG['duration_days'], 
        CONFIG['sample_interval_min'],
        start_time=CONFIG['start_date'],
        degradation_multiplier=CONFIG['degradation_multiplier'],
        include_equipment_type=True,
        maintenance_downtime_hours=24.0  # 24 hours downtime for repairs
    ):
        if record['type'] == 'failure':
            turbine_failures.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'failure_time': record['failure_time'],
                'operating_hours_at_failure': record['operating_hours_at_failure'],
                'failure_mode_code': record['failure_mode_code'],
                'state': record['state']
            })
        elif record['type'] == 'maintenance_start':
            turbine_maintenance.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'start_time': record['start_time'],
                'failure_code': record['failure_code'],
                'repaired_components': record['repaired_components'],
                'downtime_hours': record['downtime_hours']
            })
        elif record['type'] == 'telemetry':
            # Raw record for bulk DB insertion
            turbine_telemetry_raw.append(record)
            # Flattened record for DataFrame analysis
            turbine_telemetry.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'sample_time': record['sample_time'],
                'operating_hours': record['operating_hours'],
                'in_maintenance': record.get('in_maintenance', False),
                **record['state']
            })
    
    sample_count = len([r for r in turbine_telemetry if r['equipment_id'] == turbine_id])
    failure_count = len([f for f in turbine_failures if f['equipment_id'] == turbine_id])
    print(f"  GT-{turbine_id:03d}: {sample_count:,} samples, {failure_count} failures")

print(f"\nTotal turbine telemetry: {len(turbine_telemetry):,} records")
print(f"Total turbine failures: {len(turbine_failures)}")
print(f"Total turbine maintenance events: {len(turbine_maintenance)}")

In [ ]:
df = pd.DataFrame(turbine_failures)
print(df['failure_mode_code'].value_counts())

In [ ]:
print(f"\nSimulating {CONFIG['n_compressors']} Compressors...")
compressor_telemetry = []
compressor_telemetry_raw = []  # Raw records with nested state for bulk_insert
compressor_failures = []
compressor_maintenance = []

for i in range(CONFIG['n_compressors']):
    compressor_id = i + 1
    
    # Create compressor with initial health
    initial_health = generate_initial_health(CONFIG, 'compressor')
    print(f"COMP-{compressor_id:03d} initial health: {initial_health}")
    
    compressor = Compressor(
        name=f"COMP-{compressor_id:03d}",
        initial_health=initial_health,
        design_flow=random.uniform(1200, 1800),
        design_head=random.uniform(7000, 9000),
        env_model=env_model,
        enable_environmental=True,
        enable_maintenance=True,  
        enable_thermal_transients=True,
        enable_enhanced_vibration=True,
        output_mode = OutputMode.DERIVED_FEATURES
    )
    
    # Simulate with maintenance/repair after failures
    for record in simulate_equipment(
        compressor, 
        compressor_id, 
        'compressor',
        CONFIG['duration_days'], 
        CONFIG['sample_interval_min'],
        start_time=CONFIG['start_date'],
        degradation_multiplier=CONFIG['degradation_multiplier'],
        include_equipment_type=True,
        maintenance_downtime_hours=24.0  # 24 hours downtime for repairs
    ):
        if record['type'] == 'failure':
            compressor_failures.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'failure_time': record['failure_time'],
                'operating_hours_at_failure': record['operating_hours_at_failure'],
                'failure_mode_code': record['failure_mode_code'],
                'state': record['state']
            })
        elif record['type'] == 'maintenance_start':
            compressor_maintenance.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'start_time': record['start_time'],
                'failure_code': record['failure_code'],
                'repaired_components': record['repaired_components'],
                'downtime_hours': record['downtime_hours']
            })
        elif record['type'] == 'telemetry':
            # Raw record for bulk DB insertion
            compressor_telemetry_raw.append(record)
            # Flattened record for DataFrame analysis
            compressor_telemetry.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'sample_time': record['sample_time'],
                'operating_hours': record['operating_hours'],
                'in_maintenance': record.get('in_maintenance', False),
                **record['state']
            })
    
    sample_count = len([r for r in compressor_telemetry if r['equipment_id'] == compressor_id])
    failure_count = len([f for f in compressor_failures if f['equipment_id'] == compressor_id])
    print(f"  CC-{compressor_id:03d}: {sample_count:,} samples, {failure_count} failures")

print(f"\nTotal compressor telemetry: {len(compressor_telemetry):,} records")
print(f"Total compressor failures: {len(compressor_failures)}")
print(f"Total compressor maintenance events: {len(compressor_maintenance)}")

In [ ]:
df = pd.DataFrame(compressor_failures)
print(df['failure_mode_code'].value_counts())

In [ ]:
print(f"\nSimulating {CONFIG['n_pumps']} Pumps...")
pump_telemetry = []
pump_telemetry_raw = []  # Raw records with nested state for bulk_insert
pump_failures = []
pump_maintenance = []

pump_services = [
    {'name': 'Crude Booster', 'design_flow': 200, 'design_head': 100, 'density': 850},
    {'name': 'Seawater Injection', 'design_flow': 300, 'design_head': 150, 'density': 1025},
    {'name': 'Process Water', 'design_flow': 100, 'design_head': 60, 'density': 1000},
    {'name': 'Methanol Pump', 'design_flow': 50, 'design_head': 80, 'density': 790},
    {'name': 'Fire Water', 'design_flow': 400, 'design_head': 120, 'density': 1000},
]

for i in range(CONFIG['n_pumps']):
    pump_id = i + 1
    service = pump_services[i % len(pump_services)]
    
    # Create pump with initial health
    initial_health = generate_initial_health(CONFIG, 'pump')
    
    pump = Pump(
        name=f"PUMP-{pump_id:03d}",
        initial_health=initial_health,
        design_flow=service['design_flow'] * random.uniform(0.9, 1.1),
        design_head=service['design_head'] * random.uniform(0.9, 1.1),
        design_speed=3000 + random.randint(-500, 500),
        fluid_density=service['density'],
        npsh_available=random.uniform(6, 12),
        env_model=env_model,
        enable_environmental=True,  
        enable_maintenance=True,  
        enable_thermal_transients=True,
        enable_enhanced_vibration=True,
        output_mode = OutputMode.DERIVED_FEATURES
    )
    
    # Simulate with maintenance/repair after failures
    for record in simulate_equipment(
        pump, 
        pump_id, 
        'pump',
        CONFIG['duration_days'], 
        CONFIG['sample_interval_min'],
        start_time=CONFIG['start_date'],
        degradation_multiplier=CONFIG['degradation_multiplier'],
        include_equipment_type=True,
        maintenance_downtime_hours=12.0  # 12 hours downtime for pump repair
    ):
        if record['type'] == 'failure':
            pump_failures.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'failure_time': record['failure_time'],
                'operating_hours_at_failure': record['operating_hours_at_failure'],
                'failure_mode_code': record['failure_mode_code'],
                'state': record['state']
            })
        elif record['type'] == 'maintenance_start':
            pump_maintenance.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'start_time': record['start_time'],
                'failure_code': record['failure_code'],
                'repaired_components': record['repaired_components'],
                'downtime_hours': record['downtime_hours']
            })
        elif record['type'] == 'telemetry':
            # Raw record for bulk DB insertion
            pump_telemetry_raw.append(record)
            # Flattened record for DataFrame analysis
            pump_telemetry.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'sample_time': record['sample_time'],
                'operating_hours': record['operating_hours'],
                'in_maintenance': record.get('in_maintenance', False),
                **record['state']
            })
    
    if i % 5 == 4:  # Print every 5 pumps
        print(f"  PUMP-{pump_id-4:03d} to PUMP-{pump_id:03d} completed")

print(f"\nTotal pump telemetry: {len(pump_telemetry):,} records")
print(f"Total pump failures: {len(pump_failures)}")
print(f"Total pump maintenance events: {len(pump_maintenance)}")

In [ ]:
df = pd.DataFrame(pump_failures)
print(df['failure_mode_code'].value_counts())

In [ ]:
# Bulk insert all telemetry, failure, and maintenance data using optimized COPY command
db_url = os.getenv('POSTGRES_URL')
db = Database(db_url)
db.connect()

# Combine all failures and maintenance
all_failures = turbine_failures + compressor_failures + pump_failures
all_maintenance = turbine_maintenance + compressor_maintenance + pump_maintenance

print("BULK INSERTING DATA (PostgreSQL COPY)")
print(f"  Turbine records:    {len(turbine_telemetry_raw):,}")
print(f"  Compressor records: {len(compressor_telemetry_raw):,}")
print(f"  Pump records:       {len(pump_telemetry_raw):,}")
print(f"  Failure events:     {len(all_failures)}")
print(f"  Maintenance events: {len(all_maintenance)}")

try:
    t_count = bulk_insert_telemetry(db, turbine_telemetry_raw, 'turbine')
    print(f"\n  Inserted {t_count:,} turbine telemetry records")

    c_count = bulk_insert_telemetry(db, compressor_telemetry_raw, 'compressor')
    print(f"  Inserted {c_count:,} compressor telemetry records")

    p_count = bulk_insert_telemetry(db, pump_telemetry_raw, 'pump')
    print(f"  Inserted {p_count:,} pump telemetry records")

    f_count = insert_failures(db, all_failures)
    print(f"  Inserted {f_count} failure events")

    m_count = insert_maintenance(db, all_maintenance)
    print(f"  Inserted {m_count} maintenance events")

    # print(f"\nTotal: {t_count + c_count + p_count:,} telemetry + {f_count} failures + {m_count} maintenance")
except Exception as e:
    print(f"\nError during bulk insert: {e}")
    raise
finally:
    db.close()
    print("\nDatabase connection closed")

In [10]:
# Verify data in database
engine = create_engine(os.getenv('POSTGRES_URL'))

with engine.connect() as conn:
    # Telemetry counts
    print("Telemetry Records:")
    for table, label in [
        ('telemetry.gas_turbine_telemetry', 'Turbines'),
        ('telemetry.compressor_telemetry', 'Compressors'),
        ('telemetry.pump_telemetry', 'Pumps'),
    ]:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        print(f"  {label:15s} {count:>12,}")

    # Failure counts
    print("\nFailure Events:")
    for table, label in [
        ('failure_events.gas_turbine_failures', 'Turbines'),
        ('failure_events.compressor_failures', 'Compressors'),
        ('failure_events.pump_failures', 'Pumps'),
    ]:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        print(f"  {label:15s} {count:>12,}")

    # Maintenance counts
    print("\nMaintenance Events:")
    for table, label in [
        ('maintenance_events.gas_turbine_maintenance', 'Turbines'),
        ('maintenance_events.compressor_maintenance', 'Compressors'),
        ('maintenance_events.pump_maintenance', 'Pumps'),
    ]:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        print(f"  {label:15s} {count:>12,}")

    # Time range sanity check
    print("\nTime Ranges:")
    for table, label in [
        ('telemetry.gas_turbine_telemetry', 'Turbines'),
        ('telemetry.compressor_telemetry', 'Compressors'),
        ('telemetry.pump_telemetry', 'Pumps'),
    ]:
        row = conn.execute(text(
            f"SELECT MIN(sample_time), MAX(sample_time) FROM {table}"
        )).fetchone()
        if row and row[0]:
            print(f"  {label:15s} {row[0]} → {row[1]}")

    # Spot-check derived features
    print("\nDerived Features Sample (pump_telemetry):")
    rows = conn.execute(text(
        "SELECT vibration_trend_7d, speed_stability, efficiency_degradation_rate, "
        "pressure_ratio, load_factor FROM telemetry.pump_telemetry "
        "WHERE vibration_trend_7d IS NOT NULL LIMIT 3"
    )).fetchall()
    for row in rows:
        print(f"  trend={row[0]}, stability={row[1]}, deg_rate={row[2]}, "
              f"p_ratio={row[3]}, load={row[4]}")

    # Spot-check maintenance
    print("\nMaintenance Sample (pump_maintenance):")
    rows = conn.execute(text(
        "SELECT pump_id, start_time, end_time, failure_code, downtime_hours, repaired_components "
        "FROM maintenance_events.pump_maintenance LIMIT 3"
    )).fetchall()
    for row in rows:
        print(f"  pump={row[0]}, start={row[1]}, end={row[2]}, "
              f"code={row[3]}, downtime={row[4]}h, repairs={row[5]}")

engine.dispose()
print("\nSIMULATION & INGESTION COMPLETE")

Telemetry Records:
  Turbines             777,443
  Compressors          777,491
  Pumps              1,554,970

Failure Events:
  Turbines                 157
  Compressors              109
  Pumps                    564

Maintenance Events:
  Turbines                 157
  Compressors              109
  Pumps                    564

Time Ranges:
  Turbines        2025-07-01 00:00:00 → 2025-12-27 23:55:00
  Compressors     2025-07-01 00:00:00 → 2025-12-27 23:55:00
  Pumps           2025-07-01 00:00:00 → 2025-12-27 23:55:00

Derived Features Sample (pump_telemetry):
  trend=0.0, stability=0.0, deg_rate=0.0, p_ratio=1.0759, load=0.151
  trend=-0.014, stability=0.255, deg_rate=0.0013, p_ratio=1.1426, load=0.2975
  trend=-0.003, stability=0.2076, deg_rate=-0.0008, p_ratio=1.1203, load=None

Maintenance Sample (pump_maintenance):
  pump=1, start=2025-07-23 19:00:00, end=2025-07-24 07:00:00, code=F_BEARING_DRIVE_END, downtime=12.0h, repairs={'bearing_de': 0.9008}
  pump=1, start=2025-08-02 